<a href="https://colab.research.google.com/github/rezaserajian/Active-IT/blob/main/Model_1G.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ------------------------------------------------------------
# EC247 Final Project
# Based on earlier Model 1B: CNN + BiGRU architecture for EMG → QWERTY decoding
#
# This notebook uses the official emg2qwerty repository for
# dataset handling, preprocessing, and training utilities.
#
# Instead of modifying the repo pipeline, we import the repo
# modules and replace the neural network architecture.
#
# This ensures that:
#   - data preprocessing is identical to the baseline
#   - train/val/test splits remain the same
#   - results are comparable to the official baseline model
# ------------------------------------------------------------

# move into repo directory
%cd /home/jappel94/emg2qwerty

# ensure dependencies are installed (safe to run again)
!pip install -r requirements.txt
!pip install -e .

/home/jappel94/emg2qwerty


  Using cached https://github.com/kpu/kenlm/archive/master.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Obtaining file:///home/jappel94/emg2qwerty
  Preparing metadata (setup.py) ... done
  Attempting uninstall: emg2qwerty
    Found existing installation: emg2qwerty 0.1.0
    Uninstalling emg2qwerty-0.1.0:
      Successfully uninstalled emg2qwerty-0.1.0
  Running setup.py develop for emg2qwerty

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ------------------------------------------------------------
# Load dataset splits exactly as defined in the repo
# ------------------------------------------------------------

from pathlib import Path
import yaml

# repo modules for preprocessing and dataloading
from emg2qwerty.transforms import LogSpectrogram
from emg2qwerty.lightning import WindowedEMGDataModule

# ------------------------------------------------------------
# Load official train / val / test split configuration
# This file is provided in the repo and must not be modified
# ------------------------------------------------------------

with open("config/user/single_user.yaml") as f:
    cfg = yaml.safe_load(f)

# ------------------------------------------------------------
# Define path to where the HDF5 sessions are stored
# This is the only path that differs depending on your VM setup
# ------------------------------------------------------------

data_root = Path("/home/jappel94")

# ------------------------------------------------------------
# Build session lists using the official split
# ------------------------------------------------------------

train_sessions = [
    data_root / f"{item['session']}.hdf5"
    for item in cfg["dataset"]["train"]
]

val_sessions = [
    data_root / f"{item['session']}.hdf5"
    for item in cfg["dataset"]["val"]
]

test_sessions = [
    data_root / f"{item['session']}.hdf5"
    for item in cfg["dataset"]["test"]
]

# ------------------------------------------------------------
# Define the exact EMG preprocessing transform used by repo
# ------------------------------------------------------------

from emg2qwerty.transforms import ToTensor, LogSpectrogram

spec = lambda x: LogSpectrogram()(ToTensor(fields=["emg_left","emg_right"])(x))

In [ ]:
# ------------------------------------------------------------
# Create DataModule using parameters from repo configs
# (config/base.yaml + config/model/tds_conv_ctc.yaml)
# ------------------------------------------------------------

train_transform = spec
val_transform = spec
test_transform = spec

datamodule = WindowedEMGDataModule(
    window_length=8000,
    padding=(1800, 200),

    batch_size=32,
    num_workers=4,

    train_sessions=train_sessions,
    val_sessions=val_sessions,
    test_sessions=test_sessions,

    train_transform=train_transform,
    val_transform=val_transform,
    test_transform=test_transform
)

datamodule.setup()

In [ ]:
# ------------------------------------------------------------
# Custom Architecture: BiLSTM encoder replacing TDSConv
#
# Matches partner configuration:
# hidden_size = 256
# num_layers  = 3
#
# Keeps repo input/output format required for CTC.
# ------------------------------------------------------------

import torch
import torch.nn as nn

from emg2qwerty.charset import charset
from emg2qwerty.modules import SpectrogramNorm


class BiLSTMEncoder(nn.Module):

    def __init__(self, in_features):
        super().__init__()

        NUM_BANDS = 2
        ELECTRODE_CHANNELS = 16

        # Same normalization used by baseline model
        self.norm = SpectrogramNorm(
            channels=NUM_BANDS * ELECTRODE_CHANNELS
        )

        # BiLSTM encoder
        self.lstm = nn.LSTM(
            input_size=NUM_BANDS * in_features,
            hidden_size=256,
            num_layers=3,
            bidirectional=True
        )

        # classifier
        self.linear = nn.Linear(
            256 * 2,
            charset().num_classes
        )

        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, x):

        # (T, N, bands, channels, freq)
        x = self.norm(x)

        # flatten bands
        x = torch.flatten(x, start_dim=2)

        # BiLSTM
        x, _ = self.lstm(x)

        # classifier
        x = self.linear(x)

        return self.log_softmax(x)

In [ ]:
# ------------------------------------------------------------
# Lightning module replacing TDSConv encoder with BiLSTM
# Keeps training, CTC loss, and evaluation identical to repo
# ------------------------------------------------------------

import torch
import torch.nn as nn
import pytorch_lightning as pl
from torchmetrics import MetricCollection
from hydra.utils import instantiate

from emg2qwerty import utils
from emg2qwerty.metrics import CharacterErrorRates
from emg2qwerty.charset import charset
from emg2qwerty.data import LabelData


class BiLSTMCTCModule(pl.LightningModule):

    def __init__(self, encoder, optimizer, lr_scheduler, decoder):
        super().__init__()

        # store configs exactly like repo module
        self.save_hyperparameters(ignore=["encoder"])

        self.encoder = encoder

        self.ctc_loss = nn.CTCLoss(blank=charset().null_class)

        from emg2qwerty.decoder import CTCGreedyDecoder
        self.decoder = CTCGreedyDecoder()

        metrics = MetricCollection([CharacterErrorRates()])
        self.metrics = nn.ModuleDict({
            f"{phase}_metrics": metrics.clone(prefix=f"{phase}/")
            for phase in ["train", "val", "test"]
        })

    def forward(self, inputs):
        return self.encoder(inputs)

    def _step(self, phase, batch):

        inputs = batch["inputs"]
        targets = batch["targets"]
        input_lengths = batch["input_lengths"]
        target_lengths = batch["target_lengths"]

        emissions = self(inputs)

        loss = self.ctc_loss(
            log_probs=emissions,
            targets=targets.transpose(0,1),
            input_lengths=input_lengths,
            target_lengths=target_lengths
        )

        predictions = self.decoder.decode_batch(
            emissions=emissions.detach().cpu().numpy(),
            emission_lengths=input_lengths.detach().cpu().numpy()
        )

        metrics = self.metrics[f"{phase}_metrics"]

        targets_np = targets.detach().cpu().numpy()
        target_lengths_np = target_lengths.detach().cpu().numpy()

        for i in range(len(input_lengths)):
            target = LabelData.from_labels(
                targets_np[:target_lengths_np[i], i]
            )
            metrics.update(prediction=predictions[i], target=target)

        self.log(f"{phase}/loss", loss, batch_size=len(input_lengths))
        self.log_dict(metrics.compute(), batch_size=len(input_lengths))

        return loss

    def training_step(self, batch, batch_idx):
        return self._step("train", batch)

    def validation_step(self, batch, batch_idx):
        return self._step("val", batch)

    def test_step(self, batch, batch_idx):
        return self._step("test", batch)

    def configure_optimizers(self):

        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=1e-3
        )

        return optimizer

In [ ]:
# ------------------------------------------------------------
# Instantiate model using repo configs
# ------------------------------------------------------------

from omegaconf import OmegaConf

decoder_cfg = OmegaConf.load("config/decoder/ctc_greedy.yaml")

encoder = BiLSTMEncoder(
    in_features=528
)

model = BiLSTMCTCModule(
    encoder=encoder,
    optimizer=None,
    lr_scheduler=None,
    decoder=decoder_cfg
)

In [ ]:
# ------------------------------------------------------------
# Trainer setup matching repo baseline behavior
# - prints train/val metrics each epoch
# - saves checkpoints
# - runs test after training
# ------------------------------------------------------------

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

# checkpoint saving (same idea as repo config)
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",
    filename="bilstm_epoch{epoch}",
    monitor="val/CER",
    mode="min",
    save_top_k=1,
    save_last=True,
    verbose=True
)

# learning rate logging (repo uses this)
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=150,

    callbacks=[
        checkpoint_callback,
        lr_monitor
    ],

    log_every_n_steps=10,
    enable_progress_bar=True
)

# --------------------
# Train model
# --------------------
trainer.fit(
    model,
    datamodule=datamodule
)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


/opt/conda/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:604: UserWarning: Checkpoint directory checkpoints exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type          | Params
-------------------------------------------
0 | encoder  | BiLSTMEncoder | 5.9 M 
1 | ctc_loss | CTCLoss       | 0     
2 | metrics  | ModuleDict    | 0     
-------------------------------------------
5.9 M     Trainable params
0         Non-trainable params
5.9 M     Total params
23.583    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Epoch 147, global step 17760: 'val/CER' reached 37.66736 (best 37.66736), saving model to 'checkpoints/bilstm_epochepoch=147.ckpt' as top 1


Validation: 0it [00:00, ?it/s]

Epoch 148, global step 17880: 'val/CER' reached 37.54977 (best 37.54977), saving model to 'checkpoints/bilstm_epochepoch=148.ckpt' as top 1


Validation: 0it [00:00, ?it/s]

Epoch 149, global step 18000: 'val/CER' reached 37.42949 (best 37.42949), saving model to 'checkpoints/bilstm_epochepoch=149.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=150` reached.


In [ ]:
# -------------------------------------------
# Evaluate model on validation and test sets
# -------------------------------------------

print("\nRunning validation evaluation...")
val_metrics = trainer.validate(
    ckpt_path="best",
    datamodule=datamodule
)

print("\nRunning test evaluation...")
test_metrics = trainer.test(
    ckpt_path="best",
    datamodule=datamodule
)

print("\nValidation metrics:")
print(val_metrics)

print("\nTest metrics:")
print(test_metrics)

Restoring states from the checkpoint path at checkpoints/bilstm_epochepoch=149.ckpt



Running validation evaluation...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from checkpoint at checkpoints/bilstm_epochepoch=149.ckpt


Validation: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/CER          │    37.314414978027344     │
│          val/DER          │     2.696356773376465     │
│          val/IER          │    22.535966873168945     │
│          val/SER          │    12.082093238830566     │
│         val/loss          │    0.8224934935569763     │
└───────────────────────────┴───────────────────────────┘

...


Running test evaluation...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from checkpoint at checkpoints/bilstm_epochepoch=149.ckpt


Testing: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/CER          │     20.61811065673828     │
│         test/DER          │    2.2044520378112793     │
│         test/IER          │     3.674086809158325     │
│         test/SER          │    14.739572525024414     │
│         test/loss         │    0.9016359448432922     │
└───────────────────────────┴───────────────────────────┘


Validation metrics:
[{'val/loss': 0.8224934935569763, 'val/CER': 37.314414978027344, 'val/IER': 22.535966873168945, 'val/DER': 2.696356773376465, 'val/SER': 12.082093238830566}]

Test metrics:
[{'test/loss': 0.9016359448432922, 'test/CER': 20.61811065673828, 'test/IER': 3.674086809158325, 'test/DER': 2.2044520378112793, 'test/SER': 14.739572525024414}]


In [ ]:
"""
### Training Pipeline (Notebook Implementation)

**Data → Model → Loss**

```text
Raw EMG
  ↓
ToTensor
  ↓
LogSpectrogram
  ↓
SpectrogramNorm
  ↓
Flatten
  ↓
BiLSTM (hidden=256, layers=3, bidirectional)
  ↓
Linear
  ↓
LogSoftmax
  ↓
CTC Loss
  ↓
Greedy Decoder → CER metrics
```

---

### Notebook Training Pipeline

```text
single_user.yaml split
      ↓
WindowedEMGDataModule
(window_length = 8000, padding = (1800,200), batch_size = 32)
      ↓
Transforms (manually defined)
ToTensor → LogSpectrogram
      ↓
BiLSTM encoder model (defined in notebook)
      ↓
Optimizer (manually defined)
Adam(lr = 1e-3)
      ↓
PyTorch Lightning Trainer
(max_epochs = 150)
      ↓
Checkpoint selection using val/CER
      ↓
Final evaluation on test set
```

---

### Difference From Official Repo Training Script

The official repo training command

```text
python -m emg2qwerty.train user=single_user
```

builds the pipeline using **Hydra configuration files**.
Those configs automatically add several training components that are **not explicitly included in this notebook**.

**Transform pipeline**

Repo script:

```
ToTensor
→ LogSpectrogram
→ additional augmentation transforms
   (defined in repo config files)
```

Notebook:

```
ToTensor
→ LogSpectrogram
```

No augmentation transforms were added manually.

---

**Optimizer / scheduler**

Repo script:

```
Optimizer defined in Hydra config
Learning-rate scheduler also defined in config
```

Notebook:

```
Adam optimizer defined directly in code
No learning-rate scheduler used
```

---

**Trainer configuration**

Repo script:

```
Trainer parameters loaded from Hydra configs
```

Notebook:

```
Trainer instantiated manually in code
(max_epochs = 150, checkpoint callback, etc.)
```

---

### Key Point

The **dataset split, model architecture, CTC loss, decoding method, and evaluation metric (CER)** are identical.

The main difference is that the **official repo script constructs the training pipeline through Hydra configuration files**, while this notebook **manually recreates the datamodule, transforms, optimizer, and trainer in Python code**.
"""

In [ ]:
!jupyter nbconvert --to html your_notebook_name.ipynb

[NbConvertApp] WARNING | pattern 'your_notebook_name.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--exec